# s01: The Agent Loop — One Loop Is All You Need

**Harness Layer**: The Loop — the first bridge between the model and the real world.

> *"One loop & Bash is all you need"* — One tool + one loop = one Agent.

---

## The Problem

You ask the model: *"List the files in my directory and run XXX.py."*

The model can output a bash command, but once it's done outputting, it stops — it won't execute the command on its own, and it won't keep reasoning based on the result.

You could run it manually, paste the output back into the chat, and let it continue. Next command comes out, you run it again, paste it back.

Every round-trip, you're the middle layer. Automating that is what this notebook is about.

## The Solution

A `while True` loop: keep going when the model calls a tool, stop when it doesn't. The entire process hinges on two signals:

| Signal | Meaning | Loop Action |
|--------|---------|-------------|
| `stop_reason == "tool_use"` | Model raises hand: "I need a tool" | Execute → feed result back → continue |
| `stop_reason != "tool_use"` | Model says: "I'm done" | Exit loop |

The core pattern:

```
while stop_reason == "tool_use":
    response = LLM(messages, tools)
    execute tools
    append results

+----------+      +-------+      +---------+
|   User   | ---> |  LLM  | ---> |  Tool   |
|  prompt  |      |       |      | execute |
+----------+      +---+---+      +----+----+
                     ^               |
                     |  tool_result  |
                     +---------------+
                     (loop continues)
```

## Setup

First, install the required dependencies and load environment variables.

In [ ]:
# ── Setup: install dependencies ────────────────────────────
!pip install anthropic python-dotenv -q
print("Dependencies installed.")

In [1]:
# ── Environment ───────────────────────────────────────────
import os
from dotenv import load_dotenv

load_dotenv(override=True)

if os.getenv("ANTHROPIC_BASE_URL"):
    os.environ.pop("ANTHROPIC_AUTH_TOKEN", None)

print(f"ANTHROPIC_API_KEY set: {bool(os.getenv('ANTHROPIC_API_KEY'))}")
print(f"MODEL_ID: {os.getenv('MODEL_ID', '(not set)')}")
print(f"ANTHROPIC_BASE_URL: {os.getenv('ANTHROPIC_BASE_URL', '(default)')}")

ANTHROPIC_API_KEY set: True
MODEL_ID: deepseek-v4-flash
ANTHROPIC_BASE_URL: https://api.deepseek.com/anthropic


## Step 1: Tool Definition

We start with just one tool: **bash**. This is the only capability the agent has — it can run shell commands. That's enough to read files (`cat`), write files (`echo ... >`), find files (`find`), and much more.

In [2]:
# ── Tool definition: just bash ────────────────────────────
TOOLS = [
    {
        "name": "bash",
        "description": "Run a shell command.",
        "input_schema": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    }
]

print("Tool defined:", TOOLS[0]["name"])

Tool defined: bash


## Step 2: Tool Execution

The `run_bash` function executes the command the model requests. It includes:
- A **dangerous command blocklist** (preventing `rm -rf /`, `sudo`, etc.)
- A **120-second timeout**
- Output truncation to 50,000 characters

In [3]:
import subprocess


# ── Tool execution ────────────────────────────────────────
def run_bash(command: str) -> str:
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(
            command,
            shell=True,
            cwd=os.getcwd(),
            capture_output=True,
            text=True,
            timeout=120,
        )
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"
    except (FileNotFoundError, OSError) as e:
        return f"Error: {e}"

## Step 3: The Core Pattern — Agent Loop

This is the heart of everything. Under 30 lines — the minimal runnable agent harness kernel.

The model decides (whether to call a tool, which one), the harness executes (if called, run it, feed the result back).

**The flow:**
1. Send messages + tools to the LLM
2. Append the model's response
3. If the model didn't call a tool → **done**, return the final answer
4. If the model called a tool → execute it, collect results, append them back
5. Go back to step 1 (the loop continues)

In [4]:
from anthropic import Anthropic

client = Anthropic(base_url=os.getenv("ANTHROPIC_BASE_URL"))
MODEL = os.environ["MODEL_ID"]
SYSTEM = f"You are a coding agent at {os.getcwd()}. Use bash to solve tasks. Act, don't explain."


def agent_loop(messages: list):
    while True:
        # Step 2: Send messages + tools to the LLM
        response = client.messages.create(
            model=MODEL,
            system=SYSTEM,
            messages=messages,
            tools=TOOLS,
            max_tokens=8000,
        )

        # Append assistant turn
        messages.append({"role": "assistant", "content": response.content})

        # Step 3: If the model didn't call a tool, we're done
        if response.stop_reason != "tool_use":
            return

        # Step 4: Execute each tool call, collect results
        results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"\033[33m$ {block.input['command']}\033[0m")
                output = run_bash(block.input["command"])
                print(output[:200])
                results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": output,
                    }
                )

        # Step 5: Feed tool results back, loop continues
        messages.append({"role": "user", "content": results})


print("Agent loop function defined.")

Agent loop function defined.


## Step 4: Run the Agent

Now let's create the interactive REPL. Enter a query, and the agent loop will handle it — the model reasons, calls tools, gets results, and continues until it produces a final answer.

In [ ]:
# ── Interactive session ───────────────────────────────────
print("=" * 60)
print("s01: Agent Loop Interactive Mode")
print("=" * 60)
print()
print("Try these prompts:")
print("  1. 'Create a file called hello.py that prints Hello World'")  
print("  2. 'List all Python files in this directory'")  
print("  3. 'What is the current git branch?'")
print()

history = []
while True:
    try:
        query = input("\033[36ms01 >> \033[0m")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() in ("q", "exit", ""):
        break
    history.append({"role": "user", "content": query})
    agent_loop(history)
    # Print the model's final text response
    response_content = history[-1]["content"]
    if isinstance(response_content, list):
        for block in response_content:
            if getattr(block, "type", None) == "text":
                print(block.text)
    print()

print("Bye!")

s01: Agent Loop Interactive Mode

Try these prompts:
  1. 'Create a file called hello.py that prints Hello World'
  2. 'List all Python files in this directory'
  3. 'What is the current git branch?'



## What Just Happened?

Watch for two behaviors:
- **`stop_reason == "tool_use"`**: The model called bash → we executed it, fed the result back, and the loop continued
- **`stop_reason != "tool_use"`**: The model produced a final text answer → the loop exited

Under 30 lines — that's the minimal runnable agent harness kernel. It's not intelligence itself, but the smallest runtime framework that lets the model keep acting.

## What's Next

Right now the model only has bash — reading files requires `cat`, writing files requires `echo ... >`, finding files requires `find`. Ugly and error-prone.

**→ s02 Tool Use**: What happens when we give it 5 proper tools? Will the model call multiple tools at once? Will parallel tool executions step on each other?